In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)





In [ ]:
# Load Data
# ============================================
print("\n=== Loading Data ===")
train_df = pd.read_csv('/kaggle/input/introduction-to-deep-learning-competition/train.csv')
test_df = pd.read_csv('/kaggle/input/introduction-to-deep-learning-competition/test.csv')
print(f"Training: {train_df.shape}, Test: {test_df.shape}")




In [ ]:
# Prepare Data - Simple and Clean
# ============================================
print("\n=== Preparing Data ===")
X_train_full = train_df.drop('label', axis=1).values
y_train_full = train_df['label'].values

# Normalize
X_train_full = X_train_full / 255.0
X_train_full = X_train_full.reshape(-1, 28, 28, 1)

# Split into train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.15,
    random_state=42,
    stratify=y_train_full
)

print(f"Training: {X_train.shape}, Validation: {X_val.shape}")





In [ ]:
# Simple Effective Model
# ============================================
print("\n=== Building Model ===")

keras.backend.clear_session()

model = keras.Sequential([
    # Data Augmentation - MODERATE
    layers.RandomRotation(0.10),
    layers.RandomTranslation(0.10, 0.10),
    
    # Block 1
    layers.Conv2D(32, 5, padding='same', activation='relu', input_shape=(28, 28, 1)),
    layers.BatchNormalization(),
    layers.Conv2D(32, 5, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),
    
    # Block 2
    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),
    
    # Block 3
    layers.Conv2D(128, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),
    
    # Dense
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model.summary()


# ============================================
# Compile - Keep It Simple
# ============================================
print("\n=== Compiling Model ===")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


# ============================================
# Simple Callbacks
# ============================================
callbacks_list = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=20,
        mode='max',
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.5,
        patience=7,
        mode='max',
        min_lr=0.00001,
        verbose=1
    )
] 
# Train
# ============================================
print("\n=== Training Model ===")
print("This will take 30-60 minutes")

history = model.fit(
    X_train, y_train,
    batch_size=128,
    epochs=100,
    validation_data=(X_val, y_val),
    callbacks=callbacks_list,
    verbose=1
)


In [ ]:
# Evaluate
# ============================================
print("\n=== Evaluation ===")

train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

print(f"\nTraining Accuracy: {train_acc:.4f} ({train_acc*100:.2f}%)")
print(f"Validation Accuracy: {val_acc:.4f} ({val_acc*100:.2f}%)")
print(f"Overfitting gap: {(train_acc - val_acc)*100:.2f}%")

if val_acc >= 0.94:
    print("🎉 Excellent! 94%+ achieved!")
elif val_acc >= 0.93:
    print("✅ Good! 93%+ achieved!")
elif val_acc >= 0.92:
    print("👍 Decent! 92%+ achieved!")
else:
    print("📊 Room for improvement")


# ============================================
# Plot
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(history.history['accuracy'], label='Train', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
axes[0].axhline(y=0.94, color='red', linestyle='--', alpha=0.3, label='94% Target')
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'], label='Train', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2)
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


    
   

In [ ]:
# Test Data
# ============================================
print("\n=== Preparing Test Data ===")

test_ids = test_df['id'].values
X_test = test_df.drop('id', axis=1).values
X_test = X_test / 255.0
X_test = X_test.reshape(-1, 28, 28, 1)


# ============================================
# Predictions with Simple TTA
# ============================================
print("\n=== Making Predictions ===")

# TTA: Make predictions 3 times and average
all_preds = []
for i in range(3):
    print(f"Prediction pass {i+1}/3...")
    preds = model.predict(X_test, batch_size=128, verbose=0)
    all_preds.append(preds)

predictions = np.mean(all_preds, axis=0)
predicted_labels = np.argmax(predictions, axis=1)


# ============================================
# Submission
# ============================================
print("\n=== Creating Submission ===")

submission = pd.DataFrame({
    'id': test_ids,
    'label': predicted_labels
})

submission.to_csv('submission.csv', index=False)

print("\n✅ submission.csv saved!")
print(f"Total predictions: {len(submission)}")
print("\nFirst 10 predictions:")
print(submission.head(10))

In [ ]:
# Visualize
# ============================================
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

plt.figure(figsize=(15, 6))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_test[i].reshape(28, 28), cmap='gray')
    conf = predictions[i][predicted_labels[i]] * 100
    plt.title(f"{class_names[predicted_labels[i]]}\n{conf:.1f}%", fontsize=9)
    plt.axis('off')
plt.tight_layout()
plt.show()


# ============================================
# Summary
# ============================================
print("\n" + "="*70)
print("SIMPLE SOLUTION - WHAT MAKES IT WORK:")
print("="*70)
print("1. No fancy optimizers - just Adam")
print("2. No complex schedules - just ReduceLROnPlateau")
print("3. Moderate augmentation (10%)")
print("4. Standard CNN architecture")
print("5. Proper dropout (0.25-0.5)")
print("6. 15% validation split")
print("7. Simple TTA (3x)")
print(f"\nResult: {val_acc*100:.2f}%")
print("="*70)